# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damlablgc/flyrankinternship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a two-stage task: first classification (is this page declining or not — is_declining_label), then that classification's probability feeds into a ranking/scoring step (rank pages by how likely/severely they're declining, so the team knows which to review first). The final output is a rank, but the classification step underneath is what makes the ranking meaningful, it's not an arbitrary sort, it's a sort backed by a learned probability.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.environ.get("HF_TOKEN")
print("Token loaded:", HF_TOKEN is not None and len(HF_TOKEN) > 0)

Token loaded: True


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [5]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_65de48885f4ef01b,content_66925e4c50aaf7eb,367.0,433.0,1.0,12.833858
1,client_65de48885f4ef01b,content_03c6e7b1aa06065f,0.0,149.0,0.0,NaN
2,client_65de48885f4ef01b,content_0231cdd0a29cabe6,3663.0,3303.0,21.0,6.197005
3,client_65de48885f4ef01b,content_9ff3c61abaea8ee2,27.0,153.0,0.0,25.986111
4,client_65de48885f4ef01b,content_372109cca71d763f,23.0,118.0,0.0,6.404762


In [6]:
data = features.copy()
data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

print(f"Total content items: {len(data)}")
print(f"Declining (>20% drop): {data['is_declining'].sum()} ({data['is_declining'].mean()*100:.1f}%)")

data[['client_hash_id', 'content_hash_id', 'imp_prev30', 'imp_last30', 'is_declining']].head(10)

Total content items: 111247
Declining (>20% drop): 71547 (64.3%)


,client_hash_id,content_hash_id,imp_prev30,imp_last30,is_declining
0,client_65de48885f4ef01b,content_66925e4c50aaf7eb,433.0,367.0,0
1,client_65de48885f4ef01b,content_03c6e7b1aa06065f,149.0,0.0,1
2,client_65de48885f4ef01b,content_0231cdd0a29cabe6,3303.0,3663.0,0
3,client_65de48885f4ef01b,content_9ff3c61abaea8ee2,153.0,27.0,1
4,client_65de48885f4ef01b,content_372109cca71d763f,118.0,23.0,1
5,client_d211cb07b9059bab,content_f7c05a082b0a4460,181.0,303.0,0
6,client_795153d5b7850ccf,content_ed983ff2209d79ed,175.0,93.0,1
7,client_cd12bcfd98942aa1,content_d25236d6a60e0f54,120.0,0.0,1
8,client_cd12bcfd98942aa1,content_4f4bb1d64f510851,250.0,3.0,1
9,client_cd12bcfd98942aa1,content_0f6e37e5132c7227,217.0,0.0,1


**What I would predict:** Whether a page's search impressions decline by more than 20% in the most recent 30-day window compared to the prior 30-day window (`imp_last30 < 0.8 × imp_prev30`), calculated directly from the warehouse's daily fact table (`fact_content_daily_performance`).

**Where it comes from:** This is a defined rule (a 20% threshold I chose) applied to observed outcomes, real daily GSC impressions, not a business decision flag. Unlike the starter CSV's `trend_direction`, which is pre-aggregated inside a single 90-day snapshot, this version is built from real daily granularity: features come from the `prev30` window, and the label outcome comes from the separate, non-overlapping `last30` window — so the feature window never overlaps the window the label is drawn from.

**Result on the warehouse data:** Out of 111,247 content items with enough prior history (imp_prev30 ≥ 100), 71,547 (64.3%) showed this kind of decline in the most recent window — confirming this isn't a rare pattern, it's the majority case, and worth building a system around.

**Honest limitation:** This is still a same-panel comparison (last 30 days vs. prior 30 days of the same snapshot), not a true "predict the future before it happens" setup — a stronger version would train on an even earlier window and validate against a truly held-out future period the model never saw during training.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

I would defend **Precision@K** (with K matched to the content team's real review capacity, e.g. top 50) as my main metric. The team can only realistically review a limited number of pages each cycle, not all 111,247 candidates. Precision@K tells us directly: of the top K pages the model recommends reviewing first, how many were actually correct? That matches exactly how the output will be used, unlike a generic metric like accuracy or ROC AUC, which measures overall separation but doesn't reflect the team's real capacity.

I haven't yet trained a model on this new warehouse-based label (Section 2) — so far I only have the starter-CSV pipeline's numbers, using the simpler same-window proxy label (`trend_direction`): baseline precision@50 = 0.24 (~12 correct out of 50), random forest precision@50 = 0.74 (~37 correct out of 50). "Good" means beating the baseline by a meaningful margin at this specific K. My next step is re-running this comparison with the warehouse-based label to see if the same gap holds.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nOne row = one content item (a single page), identified by content_id.")
print(f"Unique content_id count: {df['content_id'].nunique()}")

df[["content_id", "client_id", "content_type", "impressions_90d", "trend_direction"]].head()

Shape: 30000 rows, 44 columns

One row = one content item (a single page), identified by content_id.
Unique content_id count: 30000


,content_id,client_id,content_type,impressions_90d,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,down


One row = **one content item (a single page)**, uniquely identified by `content_id`. The dataframe has 30,000 rows and 30,000 unique `content_id` values, confirming no duplicates each page appears exactly once, with its own 90-day aggregated metrics (impressions, clicks, sessions, etc.) and its own `trend_direction` label.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (if-statement) can only check conditions one at a time, in a rigid order, e.g. "IF days_since_last_update >= 180 AND impressions_90d >= 500 THEN flag it." On the starter-CSV pipeline, my random forest's feature importance showed the real signal comes from many features weighted together (`days_with_impressions`, `log_impressions_90d`, `avg_position`, `content_age_days`, and more), each contributing a different amount, and their combination mattered more than any single threshold (baseline 0.627 AUC vs. random forest 0.750 AUC).

The warehouse data makes this even clearer: even a single simple threshold rule (20% drop) already flags 64.3% of pages as declining — meaning a plain rule alone barely narrows anything down for the content team. Writing a fixed rule that separates "genuinely worth reviewing" from "just noisy month-to-month variation" would require guessing every relevant combination of signals (volume, position, query concentration, client history depth, etc.) by hand, and it would get more fragile with every extra condition added. That's exactly the kind of pattern-weighing task a model is built for, and it's what I plan to test directly once I train a model on the warehouse-based label instead of just the rule-based one.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.